# Exploring a press corpus and comparing tokenizers

The corpus of these labs: every 1914 issue of two French daily newspapers, *L'Humanité* (360 issues) and *Le Figaro* (24 issues), digitized by the French national library (BnF). The text comes from raw optical character recognition (OCR): it has the flaws of a real company corpus. The texts are in French; the code works the same on any language.

1. Load the corpus and measure its statistics
2. Find and fix the OCR's flaws, then split into passages
3. Compare three tokenizers, and estimate what processing the corpus costs

## Data

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh french-press-1914
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/massive-intents/fr-test.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/massive-intents/en-test.parquet
!bash /tmp/ghsparse.sh dictionary-fr
%pip install -q chunspell

## 1. Loading the corpus

Each issue is a JSON file, `french-press-1914/<newspaper>/1914/<YYYYMMDD>/<YYYYMMDD>.metadata.fulltext.json`, with three fields of interest:

- `title`: the newspaper's name, in a list (`content["title"][0]`)
- `date`: the publication date, in a list (`content["date"][0]`)
- `contentAsText`: the text, one string per page, lines separated by `\n`

Build a `pandas.DataFrame` named `issues`, one row per issue, with the columns `newspaper`, `date` (as a date) and `text` (the pages joined with `\n`). Show the number of issues per newspaper and the distribution of the number of words per issue.

**Useful documentation**

- [`Path.rglob`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.rglob)
- [`Path.read_text`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text)
- [`json.loads`](https://docs.python.org/3/library/json.html#json.loads)
- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
- [`Series.str.split`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html)
- [`Series.describe`](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `Path("french-press-1914").rglob("*.json")`: every JSON file, subfolders included (sorted with `sorted`)
- `path.read_text(encoding="utf8")`, then `json.loads(...)`
- `content["title"][0]`, `content["date"][0]` and `"\n".join(content["contentAsText"])`: each issue\'s newspaper, date and text
- `pd.DataFrame(rows, columns=[...])` from a list of tuples; `pd.to_datetime` for the date column
- `issues.newspaper.value_counts()`; `issues.text.str.split().str.len().describe()`: the distribution of the number of words (`str.split()` with no argument splits into words)

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path("french-press-1914").rglob("*.json")):
  content = json.loads(path.read_text(encoding="utf8"))
  rows.append(
    (..., ..., ...)
  )  # the newspaper, the date, and the text (the pages joined with "\n")
issues = pd.DataFrame(rows, columns=[..., ..., ...])
issues["date"] = ...  # as a date

print(...)  # the number of issues per newspaper
...  # the distribution of the number of words per issue
```

### Solution

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path("french-press-1914").rglob("*.json")):
  content = json.loads(path.read_text(encoding="utf8"))
  rows.append(
    (content["title"][0], content["date"][0], "\n".join(content["contentAsText"]))
  )
issues = pd.DataFrame(rows, columns=["newspaper", "date", "text"])
issues["date"] = pd.to_datetime(issues["date"])

print(issues.newspaper.value_counts())
issues.text.str.split().str.len().describe()

### Counting words

Count the corpus's word forms, lowercased, with the regular expression `WORD` below and a `collections.Counter`. Show:

- the total number of words and the number of distinct forms
- the share of forms that appear only once (*hapax legomena*)
- the 20 most frequent forms

**Useful documentation**

- [`re.Pattern.findall`](https://docs.python.org/3/library/re.html#re.Pattern.findall)
- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)
- [`Counter.most_common`](https://docs.python.org/3/library/collections.html#collections.Counter.most_common)

In [ ]:
import re

WORD = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)?")

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `WORD.findall(text)`: the list of every word in a text; `.lower()` for lowercase
- `Counter(...)` can be built straight from a generator of words
- the total: `sum(counts.values())`; the hapax legomena: the forms whose count is 1
- `counts.most_common(20)`: the 20 most frequent forms

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from collections import Counter

counts = Counter(
  ... for text in issues.text for word in WORD.findall(text)
)  # each word, lowercased
total = sum(counts.values())
hapax = sum(1 for count in counts.values() if ...)
print(
  f"{total:,} words, {len(counts):,} forms, {hapax / len(counts):.0%} hapax legomena"
)
counts.most_common(...)
```

#### Solution

In [ ]:
from collections import Counter

counts = Counter(word.lower() for text in issues.text for word in WORD.findall(text))
total = sum(counts.values())
hapax = sum(1 for count in counts.values() if count == 1)
print(
  f"{total:,} words, {len(counts):,} forms, {hapax / len(counts):.0%} hapax legomena"
)
counts.most_common(20)

### Zipf's law

Plot each form's frequency against its rank, on log scales, with Zipf's law's line: a constant divided by the rank, the constant being the median of rank × frequency between ranks 10 and 10,000 (anchored on the first word, the line would run below the points). Three quarters of the forms are hapax legomena: show fifteen of them, drawn at random with `random.sample` after setting a seed with `random.seed` (the same draw at every run). What do you think they are?

**Useful documentation**

- [`matplotlib.pyplot.loglog`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.loglog.html)
- [`sorted`](https://docs.python.org/3/library/functions.html#sorted)
- [`numpy.arange`](https://numpy.org/doc/stable/reference/generated/numpy.arange.html)
- [`numpy.median`](https://numpy.org/doc/stable/reference/generated/numpy.median.html)
- [`random.seed`](https://docs.python.org/3/library/random.html#random.seed)
- [`random.sample`](https://docs.python.org/3/library/random.html#random.sample)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `sorted(counts.values(), reverse=True)`: the frequencies, from largest to smallest; `np.arange(1, n + 1)`: the ranks, from 1
- `plt.loglog(ranks, frequencies, label="...")`: both axes on a logarithmic scale; `np.median(frequencies[9:10000] * ranks[9:10000])`: the constant, between ranks 10 and 10,000 (indices start at 0); Zipf's line is drawn with `"--"` and `constant / ranks`
- `plt.xlabel("Rank")`, `plt.ylabel("Occurrences")`, `plt.legend()`, `plt.show()`
- the hapax legomena: the forms of `counts` whose count is 1; `random.seed(0)`, then `random.sample(forms, 15)`: fifteen at random, always the same ones

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
import random

import matplotlib.pyplot as plt
import numpy as np

frequencies = np.array(sorted(..., reverse=True))
ranks = np.arange(1, ... + 1)
constant = np.median(...)  # rank × frequency, from rank 10 to 10,000
plt.loglog(ranks, frequencies, label="French press of 1914")
plt.loglog(ranks, ..., "--", label="Zipf's law")  # the constant divided by the rank
plt.xlabel("Rank")
plt.ylabel("Occurrences")
plt.legend()
plt.show()

random.seed(0)
random.sample([word for word, count in counts.items() if ...], 15)  # the hapax legomena
```

#### Solution

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

frequencies = np.array(sorted(counts.values(), reverse=True))
ranks = np.arange(1, len(frequencies) + 1)
plt.loglog(ranks, frequencies, label="French press of 1914")
constant = np.median(frequencies[9:10000] * ranks[9:10000])
plt.loglog(ranks, constant / ranks, "--", label="Zipf's law")
plt.xlabel("Rank")
plt.ylabel("Occurrences")
plt.legend()
plt.show()

random.seed(0)
random.sample([word for word, count in counts.items() if count == 1], 15)

# Mostly OCR errors: misread words (« présldeht », « conjcerts »), merged ones
# (« en'lui ») or fragments; a few damaged proper nouns (« scheiidémann ») and
# genuinely rare words (« discrétionnaire »).

## 2. The OCR's flaws

Words split at a line end stayed split: « con- tinuer ». The regular expression `SPLIT` below looks for these pairs: a word, a hyphen, whitespace, then another word.

In [ ]:
SPLIT = re.compile(r"([^\W\d_]+)-\s+([^\W\d_]+)")

### Rejoining split words

Rejoining every pair would be wrong: « Sud- Ouest » must stay two words. A spell checker says whether the joined word exists: the [`chunspell`](https://github.com/cdhigh/chunspell) library gives access to Hunspell, LibreOffice's engine, here with Grammalecte's French dictionary. `Hunspell.spell(word)` is `True` for a known word. Case matters: « Clemenceau » is known, « clemenceau » isn't, so `known` gets the word as it is. A proper noun missing from the dictionary, like « Cail- laux », would stay split.

The corpus can act as a second dictionary: « Caillaux » appears 1,851 times. But a form found elsewhere isn't enough: « dela » appears 52 times, from a lost space, and nearly a third of the pairs added that way would be wrong. So we only consult it for what looks like a proper noun: a first piece with a capital initial, a second one in lowercase, at least five letters in all, and a joined form found at least twice in `counts`, which counted the text before any rejoining.

Store in `matches` every match of `SPLIT` in the corpus. Write a function `should_join(match)` that applies these two rules, then `rejoin(text)` that rejoins only those pairs. Count the pairs, those rejoined and those left as they are, and show ten of each kind, drawn at random with `random.sample` (after `random.seed`): what does the rule leave? Finally, rejoin each issue's text. The `known` function remembers its answers: the corpus has millions of words.

**Useful documentation**

- [`re.Pattern.finditer`](https://docs.python.org/3/library/re.html#re.Pattern.finditer)
- [`re.Match`](https://docs.python.org/3/library/re.html#match-objects)
- [`re.Pattern.sub`](https://docs.python.org/3/library/re.html#re.Pattern.sub)
- [`Hunspell.spell`](https://github.com/cdhigh/chunspell#spelling)
- [`str.istitle`](https://docs.python.org/3/library/stdtypes.html#str.istitle)
- [`str.islower`](https://docs.python.org/3/library/stdtypes.html#str.islower)
- [`functools.cache`](https://docs.python.org/3/library/functools.html#functools.cache)
- [`Series.map`](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)
- [`random.sample`](https://docs.python.org/3/library/random.html#random.sample)

In [ ]:
from functools import cache

from hunspell import Hunspell  # the chunspell package

spell = Hunspell("fr", hunspell_data_dir="dictionary-fr").spell


@cache
def known(word: str) -> bool:
  return spell(word)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `SPLIT.finditer(text)`: one `re.Match` object per match; `match[0]`: the whole match, `match[1]` and `match[2]`: its two pieces
- `SPLIT.sub(fix, text)`: `fix` gets each `re.Match` and returns the text that replaces it; `match[1] + match[2]` rejoins the pair, `match[0]` leaves it as it is
- `known(word)` says whether the dictionary knows a word
- `first.istitle()`: a capital initial, then lowercase; `second.islower()`: all lowercase
- `counts[joined.lower()]`: the occurrences of the joined form (`counts` is in lowercase), 0 if it's absent
- the rejoined pairs: `[match[0] for match in matches if should_join(match)]`, and the same for those left; `random.seed(0)`, then `random.sample(pairs, 10)`: ten at random, always the same ones
- `issues.text.map(rejoin)` applies the function to each text

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
matches = [match for text in issues.text for match in SPLIT.finditer(...)]


def should_join(match: re.Match[str]) -> bool:
  first, second = match[1], match[2]
  joined = first + second
  is_name = ... and ... and len(joined) >= 5  # a capital initial, then lowercase
  return known(joined) or (is_name and ...)  # at least twice in the corpus


def rejoin(text: str) -> str:
  def fix(match: re.Match[str]) -> str:
    return (
      ... if should_join(match) else ...
    )  # the rejoined word, or the match as it is

  return SPLIT.sub(fix, text)


rejoined = [match[0] for match in matches if ...]  # the pairs to rejoin
kept = [match[0] for match in matches if ...]  # those left as they are
print(f"{len(matches)} pairs, {len(rejoined)} rejoined, {len(kept)} left as they are")
random.seed(0)
print(random.sample(rejoined, ...))
print(random.sample(kept, ...))
issues["text"] = issues.text.map(...)
```

#### Solution

In [ ]:
matches = [match for text in issues.text for match in SPLIT.finditer(text)]


def should_join(match: re.Match[str]) -> bool:
  first, second = match[1], match[2]
  joined = first + second
  is_name = first.istitle() and second.islower() and len(joined) >= 5
  return known(joined) or (is_name and counts[joined.lower()] >= 2)


def rejoin(text: str) -> str:
  def fix(match: re.Match[str]) -> str:
    return match[1] + match[2] if should_join(match) else match[0]

  return SPLIT.sub(fix, text)


rejoined = [match[0] for match in matches if should_join(match)]
kept = [match[0] for match in matches if not should_join(match)]
print(f"{len(matches)} pairs, {len(rejoined)} rejoined, {len(kept)} left as they are")
random.seed(0)
print(random.sample(rejoined, 10))
print(random.sample(kept, 10))
issues["text"] = issues.text.map(rejoin)

# Rejoined: words (« étincel- lement ») and proper nouns (« Gam- betta »). Left:
# OCR noise (« potir- Cfeeiboi »), compound words (« Autriche- Hongrie »,
# « Saint- Lazare »), lost spaces (« il- fallait ») and split words that the OCR
# misread (« Fon- tienav »), which the rule can't rejoin.

### The noise rate

A text's **noise rate**: the share of its words (longer than one letter) that the dictionary doesn't know. Write a function `noise(text)`, compute it for each issue and compare the two newspapers. Show the corpus's 20 most frequent unknown forms of more than five letters (the shorter ones are mostly debris: « da », « tre »): OCR errors, proper nouns, old spellings?

Many are halves of words split by a space, without a hyphen: « gou vernement », « so cialiste ». The published passages rejoin those too, when one of the two pieces is unknown and the joined word is known.

**Useful documentation**

- [`Series.map`](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)
- [`DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [`Counter.most_common`](https://docs.python.org/3/library/collections.html#collections.Counter.most_common)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- a text's words longer than one letter: `[word for word in WORD.findall(text) if len(word) > 1]`
- the noise rate: the share of those words that `known` doesn't know; `max(len(words), 1)` avoids a division by zero
- `issues.text.map(noise)`, then `issues.groupby("newspaper").noise.describe()`: the statistics per newspaper
- `Counter(...).most_common(20)` over the unknown forms of more than five letters (`len(word) > 5`), lowercased

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
def words(text: str) -> list[str]:
  return [
    word for word in WORD.findall(text) if ...
  ]  # the words longer than one letter


def noise(text: str) -> float:
  found = words(text)
  return sum(... for word in found) / max(len(found), 1)  # the share of unknown words


issues["noise"] = issues.text.map(...)
print(issues.groupby(...).noise.describe())
Counter(
  word.lower() for text in issues.text for word in words(text) if ...
).most_common(20)
```

#### Solution

In [ ]:
def words(text: str) -> list[str]:
  return [word for word in WORD.findall(text) if len(word) > 1]


def noise(text: str) -> float:
  found = words(text)
  return sum(not known(word) for word in found) / max(len(found), 1)


issues["noise"] = issues.text.map(noise)
print(issues.groupby("newspaper").noise.describe())
Counter(
  word.lower()
  for text in issues.text
  for word in words(text)
  if len(word) > 5 and not known(word)
).most_common(20)

# Proper nouns (Caillaux, Barthou), unknown to the dictionary even rejoined, capitals
# without their accent (l'Etat, THEATRE) and halves of words (vernement).

### Splitting into passages

A whole issue is too long for a model, and mixes dozens of articles: we split it into **passages**. Write a function `passages(text)` that accumulates an issue's lines until it has at least 100 words, and cuts at the first line that ends a sentence (with `.`, `!`, `?` or `»`). How many passages do you get, and how many words long?

Then compare with `passages.parquet`, the version published for the next labs: it rejoins the pairs and measures the noise with `wordfreq`'s word list, more tolerant than our dictionary (18,980 pairs rejoined), also splits lines longer than 250 words into sentences, and drops the passages whose noise is above 0.15 and the duplicates. How many passages, how long? Compare the two distributions of the number of words: where does the gap come from? Read three published passages at random.

**Useful documentation**

- [`str.split`](https://docs.python.org/3/library/stdtypes.html#str.split)
- [`str.endswith`](https://docs.python.org/3/library/stdtypes.html#str.endswith)
- [`pandas.read_parquet`](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)
- [`Series.sample`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sample.html)
- [`Series.describe`](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `text.split("\n")`: the lines; `" ".join(line.split())` normalizes a line\'s whitespace; `line.split()` with no argument: the words
- `END = re.compile(r"[.!?»]\s*$")` and `END.search(line)`: the line ends a sentence (or `line.endswith((".", "!", "?", "»"))`)
- accumulate the lines in `current` and their words in `n`; at 100 words or more and a sentence end, append `" ".join(current)` to the list of passages and start again from zero
- your passages' number of words: `pd.Series([len(passage.split()) for passage in mine]).describe()`, to compare with `published.n_words.describe()`; for the gap, the function only cuts at a line end: how many words does an OCR line hold?
- `pd.read_parquet("passages.parquet")`, `published.newspaper.value_counts()`, `published.n_words.describe()`
- `published.text.sample(3, random_state=0)`: three random passages, always the same ones

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
END = re.compile(r"[.!?»]\s*$")


def passages(text: str) -> list[str]:
  out, current, n = [], [], 0
  for line in text.split("\n"):
    line = " ".join(line.split())
    if not line:
      continue
    current.append(line)
    n += len(line.split())
    if n >= ... and END.search(line):
      out.append(" ".join(current))
      current, n = ..., ...  # start again from zero
  return out


mine = [passage for text in issues.text for passage in passages(text)]
print(len(mine))
# your passages' number of words
print(pd.Series([... for passage in mine]).describe())

published = pd.read_parquet(...)
print(published.newspaper.value_counts())
print(published.n_words.describe())
for text in published.text.sample(3, random_state=0):
  print(text, end="\n\n")
```

#### Solution

In [ ]:
END = re.compile(r"[.!?»]\s*$")


def passages(text: str) -> list[str]:
  out, current, n = [], [], 0
  for line in text.split("\n"):
    line = " ".join(line.split())
    if not line:
      continue
    current.append(line)
    n += len(line.split())
    if n >= 100 and END.search(line):
      out.append(" ".join(current))
      current, n = [], 0
  return out


mine = [passage for text in issues.text for passage in passages(text)]
print(len(mine))
print(pd.Series([len(passage.split()) for passage in mine]).describe())

published = pd.read_parquet("passages.parquet")
print(published.newspaper.value_counts())
print(published.n_words.describe())
for text in published.text.sample(3, random_state=0):
  print(text, end="\n\n")

# Your passages: a median of 334 words, 9,003 at most, against 118 and 573 for the
# published ones. The function only cuts at a line end, and some OCR lines are whole
# paragraphs: one in ten is longer than 250 words, up to 1,664. A 1,000-word line
# isn't a passage: hence the published version, which splits these lines into
# sentences.

## 3. Comparing tokenizers

Load three tokenizers with `transformers.AutoTokenizer.from_pretrained` (the tokenizer alone downloads in seconds):

- `google-bert/bert-base-multilingual-cased` (WordPiece)
- `Qwen/Qwen3-Embedding-0.6B` (byte-level BPE; the embedding model of the next labs)
- `FacebookAI/xlm-roberta-base` (Unigram)

For each, show the tokens and the number of tokens per word of the texts in `samples`. `convert_ids_to_tokens` writes the tokens as the vocabulary does, markers included: `##` (WordPiece) says that a token continues a word, `▁` (XLM-R) that it starts one. What costs the most tokens?

Four sentences aren't enough to compare two languages. `fr-test.parquet` and `en-test.parquet`, downloaded above, hold the same 2,974 requests to a voice assistant, in French and in English (the MASSIVE dataset, which the intent classification labs use again). For each tokenizer, measure in each language the number of words and of tokens per word, then the ratio of French tokens to English ones: for the same meaning, how many more tokens does French cost, and why?

**Useful documentation**

- [`AutoTokenizer.from_pretrained`](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoTokenizer.from_pretrained)
- [`tokenizer(...)`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.__call__)
- [`tokenizer.convert_ids_to_tokens`](https://huggingface.co/docs/transformers/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.convert_ids_to_tokens)
- [`pandas.read_parquet`](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)
- [Tokenization algorithms](https://huggingface.co/docs/transformers/tokenizer_summary)

In [ ]:
samples = {
  "French": "Le conseil municipal a voté hier soir le budget de la nouvelle médiathèque.",
  "English": "The city council voted on the budget for the new media library last night.",
  "proper noun": "Clemenceau et Poincaré",
  "OCR": "Où .av b; aifn'wmant Mme Gueydan, it. o ta î - iu ei est arrivée a la barre",
}

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `AutoTokenizer.from_pretrained(name)`; store the three in a `tokenizers` dictionary (name → tokenizer): a later exercise uses it
- `tokenizer(text, add_special_tokens=False).input_ids`: the tokens' IDs, without the special tokens (`[CLS]`, `<s>`…) we don't want to count
- `tokenizer.convert_ids_to_tokens(ids)`: the tokens of a list of IDs. Qwen3 splits bytes, not characters: there `Ġ` marks the space before a word, and an accented letter, two bytes in UTF-8, is written as two characters (`Ã©` for « é »)
- tokens per word: `len(ids) / len(text.split())`
- `pd.read_parquet("fr-test.parquet").text`: the requests in French; `texts.str.split().str.len().sum()`: their number of words; their number of tokens: the sum of each request's `len(ids)`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from transformers import AutoTokenizer

tokenizers = {
  name: AutoTokenizer.from_pretrained(...)
  for name in (
    "google-bert/bert-base-multilingual-cased",
    "Qwen/Qwen3-Embedding-0.6B",
    "FacebookAI/xlm-roberta-base",
  )
}
for name, tokenizer in tokenizers.items():
  print(name)
  for label, text in samples.items():
    ids = tokenizer(text, add_special_tokens=False).input_ids
    tokens = tokenizer.convert_ids_to_tokens(...)
    print(f"  {label:11} {len(ids) / len(text.split()):.2f} tokens/word {tokens}")

massive = {
  "fr": pd.read_parquet("fr-test.parquet").text,
  "en": pd.read_parquet(...).text,
}
n_words = {language: ... for language, texts in massive.items()}  # the number of words
print(f"{n_words['fr']:,} words in French, {n_words['en']:,} in English")
for name, tokenizer in tokenizers.items():
  n_tokens = {
    language: sum(
      len(tokenizer(text, add_special_tokens=False).input_ids) for text in texts
    )
    for language, texts in massive.items()
  }
  print(
    f"{name}: {n_tokens['fr'] / n_words['fr']:.2f} tokens/word in French, "
    f"{n_tokens['en'] / n_words['en']:.2f} in English, "
    f"{... / ...:.2f} times more tokens in French"
  )
```

### Solution

In [ ]:
from transformers import AutoTokenizer

tokenizers = {
  name: AutoTokenizer.from_pretrained(name)
  for name in (
    "google-bert/bert-base-multilingual-cased",
    "Qwen/Qwen3-Embedding-0.6B",
    "FacebookAI/xlm-roberta-base",
  )
}
for name, tokenizer in tokenizers.items():
  print(name)
  for label, text in samples.items():
    ids = tokenizer(text, add_special_tokens=False).input_ids
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f"  {label:11} {len(ids) / len(text.split()):.2f} tokens/word {tokens}")

massive = {
  "fr": pd.read_parquet("fr-test.parquet").text,
  "en": pd.read_parquet("en-test.parquet").text,
}
n_words = {
  language: texts.str.split().str.len().sum() for language, texts in massive.items()
}
print(f"{n_words['fr']:,} words in French, {n_words['en']:,} in English")
for name, tokenizer in tokenizers.items():
  n_tokens = {
    language: sum(
      len(tokenizer(text, add_special_tokens=False).input_ids) for text in texts
    )
    for language, texts in massive.items()
  }
  print(
    f"{name}: {n_tokens['fr'] / n_words['fr']:.2f} tokens/word in French, "
    f"{n_tokens['en'] / n_words['en']:.2f} in English, "
    f"{n_tokens['fr'] / n_tokens['en']:.2f} times more tokens in French"
  )

# The proper noun costs the most per word (3 tokens per word for Qwen3 and XLM-R: a few
# letters each), then the noisy OCR text (its word pieces aren't in the vocabularies).
# On MASSIVE, French takes 8% more words than English, and more tokens per word with
# all three tokenizers (1.54 against 1.08 for Qwen3): for the same meaning, 1.32 times
# more tokens for BERT, 1.35 for XLM-R, 1.54 for Qwen3. The vocabularies, learned on
# corpora where English dominates, split French more. XLM-R, nearly even on the
# sentence in `samples`, isn't here: one sentence proves nothing. These figures
# aren't the ones of the course's figure, measured on other sentences.

### What the corpus costs

How many tokens (for Qwen3's tokenizer) does the published passages corpus hold? Estimate it on a sample of 2,000 passages. At $0.10 per million input tokens (the order of magnitude of a small model through an API), what does one pass of the whole corpus through an LLM cost, input alone?

An encoder limited to 512 tokens (BERT, XLM-R) only sees the first 512 of a longer text: the rest is **truncated**, often without a warning. For each of the three tokenizers, what share of the sample's passages is longer than 512 tokens? And among 500 of your passages (`mine`), drawn at random?

**Useful documentation**

- [`Series.sample`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sample.html)
- [`tokenizer(...)`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.__call__)
- [`numpy.mean`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `tokenizers["Qwen/Qwen3-Embedding-0.6B"]`: Qwen3's tokenizer, from the previous exercise
- `published.text.sample(2000, random_state=0)`: 2,000 random passages, always the same ones
- `len(tokenizer(text, add_special_tokens=False).input_ids)`: a passage's number of tokens, without the special tokens (Qwen3's tokenizer appends `<|endoftext|>` to every text); `np.mean` averages them
- `len(tokenizer(text).input_ids)`: the length the model sees, special tokens included (`[CLS]` and `[SEP]` take two of BERT's 512 positions)
- the total: the mean × the number of passages (`len(published)`); the cost: the total in millions of tokens × $0.10
- `pd.Series(mine).sample(500, random_state=0)`: 500 of your passages, always the same ones
- `np.array(lengths) > 512`: an array of booleans, whose `np.mean` is the share of `True`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
tokenizer = tokenizers["Qwen/Qwen3-Embedding-0.6B"]
sample = published.text.sample(..., random_state=0)
tokens_per_passage = np.mean(
  [len(tokenizer(text, add_special_tokens=False).input_ids) for text in ...]
)
total_tokens = tokens_per_passage * ...
print(
  f"{tokens_per_passage:.0f} tokens per passage, {total_tokens / 1e6:.1f} million in total"
)
print(f"${total_tokens / 1e6 * ...:.2f} for one pass of the corpus")

mine_sample = pd.Series(mine).sample(..., random_state=0)
for name, tokenizer in tokenizers.items():
  published_lengths = np.array([len(tokenizer(text).input_ids) for text in sample])
  mine_lengths = np.array([... for text in mine_sample])
  print(
    f"{name}: {np.mean(published_lengths > ...):.2%} of the published passages, "
    f"{np.mean(mine_lengths > ...):.0%} of yours beyond 512 tokens"
  )
```

#### Solution

In [ ]:
tokenizer = tokenizers["Qwen/Qwen3-Embedding-0.6B"]
sample = published.text.sample(2000, random_state=0)
tokens_per_passage = np.mean(
  [len(tokenizer(text, add_special_tokens=False).input_ids) for text in sample]
)
total_tokens = tokens_per_passage * len(published)
print(
  f"{tokens_per_passage:.0f} tokens per passage, {total_tokens / 1e6:.1f} million in total"
)
print(f"${total_tokens / 1e6 * 0.10:.2f} for one pass of the corpus")

mine_sample = pd.Series(mine).sample(500, random_state=0)
for name, tokenizer in tokenizers.items():
  published_lengths = np.array([len(tokenizer(text).input_ids) for text in sample])
  mine_lengths = np.array([len(tokenizer(text).input_ids) for text in mine_sample])
  print(
    f"{name}: {np.mean(published_lengths > 512):.2%} of the published passages, "
    f"{np.mean(mine_lengths > 512):.0%} of yours beyond 512 tokens"
  )

# Less than 1% of the published passages are longer than 512 tokens (0.10% with BERT,
# 0.25% with XLM-R, 0.75% with Qwen3), but six in ten of yours (58 to 61%): an encoder
# limited to 512 tokens would only read their beginning. BERT's and XLM-R's
# tokenizers say so with a warning, but cut nothing; an embedding model loaded by
# sentence-transformers truncates without a word. Hence short passages. Qwen3's tokenizer reports
# `model_max_length` = 131,072, but its model card gives the model 32,768 tokens
# (32k): the model card is what counts.